# Setup and Config

In [0]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import (
  col, lit, trim, upper, lower, when, coalesce,
  to_date, regexp_replace, regexp_extract,
  current_timestamp, split, concat
)
from pyspark.sql.types import (
  StructType,
  StructField,
  StringType,
  IntegerType,
  TimestampType,
  DoubleType,
  BooleanType,
  DateType
)

# s3 paths
PLAYERS_SOURCE_PATH = "s3://ipl-analytics-raw-data/scraped/raw_players.parquet"

# Bronze table
BRONZE_RAW_DELIVERY_TABLE = "ipl_catalog.bronze.raw_deliveries"
BRONZE_RAW_MATCH_INFO_TABLE = "ipl_catalog.bronze.raw_match_info"
BRONZE_RAW_PLAYER_TABLE = "ipl_catalog.bronze.raw_players"

# Silver tables
SILVER_DELIVERIES_TABLE = "ipl_catalog.silver.deliveries"
SILVER_MATCHES_TABLE = "ipl_catalog.silver.matches"
SILVER_PLAYERS_TABLE = "ipl_catalog.silver.players"
SILVER_VENUES_TABLE = "ipl_catalog.silver.venues"

Create catalog, schema and tables

In [0]:
# create catalog
spark.sql("CREATE CATALOG IF NOT EXISTS ipl_catalog")

# Create raw_data, BRONZE, SILVER and GOLD schemas (DATABASE)
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.raw_data")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.gold")

# Create Tables
spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_DELIVERIES_TABLE}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_MATCHES_TABLE}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_PLAYERS_TABLE}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_VENUES_TABLE}")

Team name standardisation map

In [0]:
# every kniwn variation -> standard current name
TEAM_NAME_MAP = {
  # Mumbai indians
  "mumbai indians" : "Mumbai Indians",
  "mi" : "Mumbai Indians",

  # Chennai Super Kings
  "chennai super kings" : "Chennai Super Kings",
  "csk" : "Chennai Super Kings",

  # Royal Challengers Bangalore
  "royal challengers bangalore" : "Royal Challengers Bengaluru",
  "rcb" : "Royal Challengers Benguluru",
  "royal challengers bengaluru" : "Royal Challengers Bengaluru",
  
  # Kolkata Knight Riders
  "kolkata knight riders" : "Kolkata Knight Riders",
  "kkr" : "Kolkata Knight Riders",

  # Delhi Capitals
  "delhi daredevils" : "Delhi Capitals",
  "delhi capitals" : "Delhi Capitals",

  # Rajasthan Royals
  "rajasthan royals" : "Rajasthan Royals",
  "rr" : "Rajasthan Royals",

  # Sunrisers Hyderabad
  "deccan chargers" : "Sunrisers Hyderabad",
  "sunrisers hyderabad" : "Sunrisers Hyderabad",
  "srh" : "Sunrisers Hyderabad",
  "hyderabad" : "Sunrisers Hyderabad",

  # Punjab Kings
  "kings xi punjab" : "Punjab Kings",
  "punjab kings" : "Punjab Kings",

  # Newer teams
  "lucknow super giants" : "Lucknow Super Giants",
  "lsg" : "Lucknow Super Giants",
  "gujarat titans" : "Gujarat Titans",
  "gt" : "Gujarat Titans",

  # OLD defunct teams
  "kochi tuskers kerala" : "Kochi Tuskers Kerala",
  "pune warriors" : "Pune Warriors",
  "rising pune supergiant" : "Rising Pune Supergiant",
  "rising pune supergiants" : "Rising Pune Supergiant",
  "gujarat lions" : "Gujarat Lions"
}

def standardise_team(team_col) :
  """
  Applies team name standardisation using the map above.
  Lowercase the input first to handle case variations, then maps to the standard name.
  Falls back to original value if not found in map.
  """
  result = lower(trim(team_col))
  for raw, standard in TEAM_NAME_MAP.items() :
    result = when(result == raw, standard).otherwise(result)

  return result

Clean deliveries Bronze -> Silver

In [0]:
def clean_deliveries() :
  print("Cleaning deliveries...")

  df_bronze = spark.table(BRONZE_RAW_DELIVERY_TABLE)
  print(f"Bronze row count: {df_bronze.count():,}")

  df_clean = (
    df_bronze

    # 1. remove nulls on critical columns
    .filter(col("match_id").isNotNull())
    .filter(col("innings").isNotNull())
    .filter(col("ball").isNotNull())
    .filter(col("batting_team").isNotNull())
    .filter(col("bowling_team").isNotNull())
    .filter(col("striker").isNotNull())
    .filter(col("bowler").isNotNull())

    # # 2. Standardise team names
    # .withColumn("batting_team", standardise_team(col("batting_team")))
    # .withColumn("bowling_team", standardise_team(col("bowling_team")))

    # 3. Fix season format
    .withColumn("season", col("season").substr(1, 4))

    # 4. Fix date format
    .withColumn("start_date", to_date(col("start_date"), "yyyy-MM-dd"))

    # 5. cast numeric columns
    .withColumn("innings", col("innings").cast(IntegerType()))
    .withColumn("runs_off_bat", coalesce(col("runs_off_bat").cast(IntegerType()), lit(0)))
    .withColumn("extras", coalesce(col("extras").cast(IntegerType()), lit(0)))
    .withColumn("wides", coalesce(col("wides").cast(IntegerType()), lit(0)))
    .withColumn("noballs", coalesce(col("noballs").cast(IntegerType()), lit(0)))
    .withColumn("byes", coalesce(col("byes").cast(IntegerType()), lit(0)))
    .withColumn("legbyes", coalesce(col("legbyes").cast(IntegerType()), lit(0)))

    # 6. Parse over and ball number from "1.1" format
    .withColumn("over_number", split(col("ball"), "\\.")[0].cast(IntegerType()))
    .withColumn("ball_number", split(col("ball"), "\\.")[1].cast(IntegerType()))

    # 7. clean player names - trim whitespace
    .withColumn("striker", trim(col("striker")))
    .withColumn("non_striker", trim(col("non_striker")))
    .withColumn("bowler", trim(col("bowler")))

    # 8. Handle optional wicket columns
    .withColumn("wicket_type", when(trim(col("wicket_type")) == "", None)
                .otherwise(trim(col("wicket_type"))))
    .withColumn("player_dismissed", when(trim(col("player_dismissed")) == "", None)
                .otherwise(trim(col("player_dismissed"))))

    # 9. add derived columns
    .withColumn("total_runs", col("runs_off_bat") + col("extras"))
    .withColumn("is_wicket", when(col("wicket_type").isNotNull(), True).otherwise(False))
    .withColumn("is_boundary", when(col("runs_off_bat") >= 4, True).otherwise(False))
    .withColumn("is_six", when(col("runs_off_bat") >= 6, True).otherwise(False))
    .withColumn("is_dot_ball", when((col("runs_off_bat") == 0) & (col("wides") == 0) & (col("noballs") == 0), True).otherwise(False))
    .withColumn("phase", when(col("innings") >= 3,      "super_over").when(col("over_number") <= 6, "powerplay").when(col("over_number") <= 15, "middle").otherwise("death"))

    # 10. add metadata
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source", lit("cricsheet"))

    # 11. drop bronze metadata columns
    .drop("_source_file", "_file_size", "_file_modified", "_rescued_data", "_file_match_id", "ball")

    # 12. Deduplicate
    .dropDuplicates(["match_id", "innings", "over_number", "ball_number", "striker", "bowler"])
  )

  print(f"silver row count after cleaning: {df_clean.count():,}")
  return df_clean

Clean match info Bronze -> Silver matches and venues

In [0]:
def clean_matches():
    print("Cleaning matches...")

    df_bronze = spark.table(BRONZE_RAW_MATCH_INFO_TABLE)

    # ── Pivot key-value rows into columns ──────────────────────
    # Bronze format: match_id | key        | value
    #                12345    | city       | Mumbai
    #                12345    | venue      | Wankhede Stadium
    #                12345    | winner     | Mumbai Indians

    from pyspark.sql.functions import first, max as spark_max
    from pyspark.sql import Window
    import pyspark.sql.functions as F

    # 1. Define the Window
    # We partition by match_id and key so that the row numbering 
    # restarts for every new match and every new attribute.
    window_spec = Window.partitionBy("match_id", "key").orderBy("value")

    # 2. Prepare the Keys
    # We use .isin() to target both 'team' and 'umpire' simultaneously.
    # This transforms 'team' -> 'team1', 'team2' and 'umpire' -> 'umpire1', 'umpire2'.
    df_prepared = df_bronze.withColumn(
        "key", 
        F.when(
            F.col("key").isin("team", "umpire"), 
            F.concat(F.col("key"), F.row_number().over(window_spec))
        ).otherwise(F.col("key"))
    )

    df_pivot = (
        df_prepared
        .groupBy("match_id")
        .pivot("key", [
            "city", "venue", "date", "season",
            "team1", "team2",
            "toss_winner", "toss_decision",
            "winner", "outcome", "winner_runs", "winner_wickets",
            "player_of_match", "umpire1", "umpire2",
            "method", "eliminator", "event", "match_number", "balls_per_over"
        ])
        .agg(first("value"))
    )

    df_clean = (
        df_pivot

        # ── 1. Remove nulls on critical columns ───────────────
        .filter(col("match_id").isNotNull())
        .filter(col("team1").isNotNull())
        .filter(col("team2").isNotNull())

        # # ── 2. Standardise team names ──────────────────────────
        # .withColumn("team1",        standardise_team(col("team1")))
        # .withColumn("team2",        standardise_team(col("team2")))
        # .withColumn("toss_winner",  standardise_team(col("toss_winner")))
        # .withColumn("winner",       standardise_team(col("winner")))

        # ── 3. Fix date format ─────────────────────────────────
        .withColumn("match_date",
                    to_date(col("date"), "yyyy/MM/dd"))

        # ── 4. Fix season format ───────────────────────────────
        .withColumn("season",
                    col("season").substr(1, 4))

        # ── 5. Clean venue and city ────────────────────────────
        .withColumn("venue", trim(col("venue")))
        .withColumn("city",  trim(col("city")))

        # ── 6. Derive result from outcome key ─────────────────────
        # outcome = winner    → match completed with a winner
        # outcome = tie       → match tied (no winner)
        # outcome = no result → abandoned/weather
        # outcome = cancelled → cancelled before start
        .withColumn("result",
                    when(lower(col("outcome")) == "winner",
                         "completed")
                    .when(lower(col("outcome")) == "tie",
                         "tie")
                    .when(lower(col("outcome")) == "no result",
                         "no_result")
                    .when(lower(col("outcome")) == "cancelled",
                         "cancelled")
                    # Fallback — infer from winner column
                    .when(col("outcome").isNull() &
                          col("winner").isNotNull(),
                         "completed")
                    .when(col("outcome").isNull() &
                          col("winner").isNull(),
                         "no_result")
                    .otherwise("completed"))

        # ── 7. Derive result_margin ───────────────────────────────
        # winner_runs    = 47 → won by 47 runs
        # winner_wickets = 7  → won by 7 wickets
        # both null           → tie or no result
        .withColumn("result_margin",
                    when(col("winner_runs").isNotNull(),
                         col("winner_runs").cast(IntegerType()))
                    .when(col("winner_wickets").isNotNull(),
                         col("winner_wickets").cast(IntegerType()))
                    .otherwise(None))

        # ── 8. Standardise toss decision ──────────────────────
        .withColumn("toss_decision",
                    when(lower(col("toss_decision")) == "bat",   "bat")
                    .when(lower(col("toss_decision")) == "field", "field")
                    .otherwise(None))
        
        # ── Derive margin_type ─────────────────────────────────
        # tells whether margin is runs or wickets
        # used in Gold layer match_summary and team_win_rates
        .withColumn("margin_type",
                    when(col("winner_runs").isNotNull(),
                         "runs")
                    .when(col("winner_wickets").isNotNull(),
                         "wickets")
                    .otherwise(None))

        # ── DLS method flag ────────────────────────────────────
        # True if Duckworth-Lewis-Stern method was applied
        .withColumn("is_dls",
                    when(col("method") == "D/L", True)
                    .otherwise(False))

        # ── Playoff match flag ─────────────────────────────────
        # eliminator key exists only for knockout matches
        .withColumn("is_playoff",
                    when(col("eliminator").isNotNull(), True)
                    .otherwise(False))
        # ── 9. Add derived columns ─────────────────────────────
        .withColumn("toss_winner_won",
                    when(col("toss_winner") == col("winner"), True)
                    .otherwise(False))
        .withColumn("loser",
                    when(col("winner") == col("team1"), col("team2"))
                    .when(col("winner") == col("team2"), col("team1"))
                    .otherwise(None))

        # ── 10. Add metadata ───────────────────────────────────
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source", lit("cricsheet"))

        # ── 11. Drop raw columns no longer needed ──────────────
        .drop("date", "outcome",
              "winner_runs", "winner_wickets",
              "method", "eliminator")

        # ── 12. Deduplicate on match_id ────────────────────────
        .dropDuplicates(["match_id"])
    )

    # print(f"Silver matches count: {df_clean.count():,}")
    print("Matches cleaning plan built — will execute on write")
    return df_clean


Extract venues from matches

In [0]:
def clean_venues(df_matches_clean):
    """
    Venues are extracted from the cleaned matches DataFrame.
    No separate Bronze source — venues are derived from match info.
    """
    print("Extracting venues...")

    df_venues = (
        df_matches_clean
        .select("venue", "city")
        .filter(col("venue").isNotNull())

        # ── Clean venue names ──────────────────────────────────
        .withColumn("venue", trim(col("venue")))
        .withColumn("city",  trim(col("city")))

        # ── Fix null cities where we can infer from venue ─────
        .withColumn("city",
                    when(col("city").isNull() &
                         col("venue").contains("Wankhede"),    "Mumbai")
                    .when(col("city").isNull() &
                         col("venue").contains("Eden Gardens"), "Kolkata")
                    .when(col("city").isNull() &
                         col("venue").contains("Chinnaswamy"),  "Bengaluru")
                    .when(col("city").isNull() &
                         col("venue").contains("Feroz Shah"),   "Delhi")
                    .otherwise(col("city")))

        # ── One row per venue ──────────────────────────────────
        .dropDuplicates(["venue"])

        # ── Add metadata ───────────────────────────────────────
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source", lit("cricsheet"))
    )

    print(f"Unique venues: {df_venues.count()}")
    return df_venues



# Silver layer for players table

In [0]:
silver_players_schema = StructType([
    StructField("player_id", StringType(), False), # cricinfo unique ID
    StructField("player_name", StringType(), False),
    StructField("team", StringType(), False),
    StructField("season", StringType(), False),
    StructField("cricinfo_url", StringType(), True),
    StructField("valid_from", DateType(), False), # season start date
    StructField("valid_to", DateType(), False), # 9999-12-31 if active
    StructField("is_current", BooleanType(), False), # True = active record
    StructField("_source", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
])

In [0]:
def extract_player_id(url_col) :
  """
  Extact numeric ID from cricinfo URL.
  '/cricketers/rohit-sharma-34102' -> '34102'
  """
  return regexp_extract(url_col, r"-(\d+)$", 1)

In [0]:
def season_start_date(season_col) :
    """
    convert season string to season start date.
    '2022'-> 2022-03-01(appropriate IPL season start)
    """
    return to_date(concat(season_col, lit("-03-01")), "yyyy-MM-dd")

In [0]:
def prepare_incoming(bronze_df) :
    """
    Clean and shape Bronze player data for SCD2 merge.
    Data cleaning + Schema Enforcement
    """
    return (
        bronze_df
        .filter(col("player_name").isNotNull())
        .filter(col("cricinfo_url").isNotNull())
        .withColumn("player_id", extract_player_id(col("cricinfo_url")))
        .withColumn("season", col("season").substr(1, 4))
        .withColumn("valid_from", season_start_date(col("season")))
        .withColumn("valid_to", to_date(lit("9999-12-31"), "yyyy-MM-dd"))
        .withColumn("is_current", lit(True))
        .withColumn("_ingested_at", current_timestamp())
        # duplicate - keep one record pe player per season
        .dropDuplicates(subset = ["player_id", "season"])
        .select(
            "player_id",
            "player_name",
            "team",
            "season",
            "cricinfo_url",
            "valid_from",
            "valid_to",
            "is_current",
            "_source",
            "_ingested_at"
        )
    )

SCD2 merge function(core logic)

In [0]:
def apply_scd2(incoming_df) :
    """
    Applies SCD2 merge to the Silver players Delta table.
    For each incoming player record:
    - If player is NEW          -> INSERT as active record
    - If player's team CHANGED  -> close old record + INSERT new active record
    - If player's team UNCHANGED-> do nothing (no-op)
    """

    # Create table if first run
    table_exists = spark.catalog.tableExists(SILVER_PLAYERS_TABLE)

    if not table_exists :
        print("Creating Silver players table for the first time...")
        empty_df = spark.createDataFrame([], silver_players_schema)
        (empty_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(SILVER_PLAYERS_TABLE)
        )

    silver_table = DeltaTable.forName(spark, SILVER_PLAYERS_TABLE)

    # --Step 1: Close old records where team has changed--
    # Join incoming against current active Silver records
    # Find rows where player_id matches but team is different
    silver_table.alias("silver").merge(
        source = incoming_df.alias("incoming"),
        condition = """
            silver.player_id = incoming.player_id
            AND silver.is_current = true
            AND silver.team != incoming.team
        """
    ).whenMatchedUpdate(set = {
        # Close the old record: set valid_to to day before new season
        "valid_to" : "date_sub(incoming.valid_from, 1)",
        "is_current" : "false"
    }).execute()

    print("Step 1 complete - closed stale actice records")

    # --Step 2: Insert new records where no active record exists -- 
    # This handles both:
    #   a) Brand new players (never seen before)
    #   b) Players whose old records was just closed in Step 1
    silver_table.alias("silver").merge(
        source = incoming_df.alias("incoming"),
        condition = """
            silver.player_id = incoming.player_id
            AND silver.season = incoming.season
        """
    ).whenNotMatchedInsert(values = {
        "player_id" : "incoming.player_id",
        "player_name" : "incoming.player_name",
        "team" : "incoming.team",
        "season" : "incoming.season",
        "cricinfo_url" : "incoming.cricinfo_url",
        "valid_from" : "incoming.valid_from",
        "valid_to" : "incoming.valid_to",
        "is_current" : "incoming.is_current",
        "_source" : "incoming._source",
        "_ingested_at" : "incoming._ingested_at"
    }).execute()

    print("Step 2 complete - inserted new/transferred player records")


Run the full silver load

In [0]:
def run_silver_players() :
  # Load from Bronze
  bronze_df = spark.table(BRONZE_RAW_PLAYER_TABLE)

  # Prepare incoming
  incoming_df = prepare_incoming(bronze_df)
  print(f"Incoming records: {incoming_df.count()}")

  # Apply SCD2
  apply_scd2(incoming_df)

  # Optimize Silver table
  spark.sql(f"OPTIMIZE {SILVER_PLAYERS_TABLE} ZORDER BY (player_id, season)")

  print("Silver players load complete")

# run_silver_players()

Verify the SCD2 output

In [0]:
df_silver = spark.table(SILVER_PLAYERS_TABLE)

# total records (will be > number of players due to transfers)
print(f"Total silver rows: {df_silver.count()}")

# Players with multiple records = transferred players
from pyspark.sql.functions import count as cnt

transfers = (
  df_silver
    .groupBy("player_id", "player_name")
    .agg(cnt("*").alias("num_records"))
    .filter(col("num_records") > 1)
    .orderBy(col("num_records").desc())
)
print("Players with most transfers:")
transfers.show(10)

# verify only one active records per player
active_counts = (
  df_silver
    .filter(col("is_current") == True)
    .groupBy("player_id")
    .agg(cnt("*").alias("active_records"))
    .filter(col("active_records") > 1)
)
assert active_counts.count() == 0, "ERROR: Multiple active records for some players!"
print("Assertion passed - every player has exactly one active record")

# view full history for a specific player
(df_silver
    .filter(col("player_name").contains("Sharma"))
    .select("player_name", "team", "season", "valid_from", "valid_to", "is_current")
    .orderBy("player_name", "valid_from")
    .show(truncate = False)
)

Write all four silver tables

In [0]:
def write_silver_table(df, table_name, mode = 'overwrite') :
    """
    Generic function to write a DataFrame to a Silver Delta table.
    Uses overwrite for full refresh tables (matches, venues).
    Uses append for large tables (deliveries).
    """
    (df.write
     .format("delta")
     .mode(mode)
     .option("mergeSchema", "true")
     .option("overwriteSchema", "true" if mode == "overwrite" else "false")
     .saveAsTable(table_name)
    )

    count = spark.table(table_name).count()
    print(f"Written {count:,} rows -> {table_name}")

def run_silver_transforms() :
    """
    Master function — runs all four Silver table transforms in order.
    Call this function to run the full Week 3 pipeline."""

    # 1. Deliveries
    df_deliveries = clean_deliveries()
    write_silver_table(
        df_deliveries,
        SILVER_DELIVERIES_TABLE,
        mode = "overwrite"
    )

    # # 2. Matches
    df_matches = clean_matches()
    write_silver_table(
        df_matches,
        SILVER_MATCHES_TABLE,
        mode = "overwrite"
    )

    # 3. Venues
    df_venues = clean_venues(df_matches)
    write_silver_table(
        df_venues,
        SILVER_VENUES_TABLE,
        mode = "overwrite"
    )

    # 4. Players - SCD2-------------------------------------DONE
    bronze_players = spark.table(BRONZE_RAW_PLAYER_TABLE)
    incoming_df = prepare_incoming(bronze_players)
    apply_scd2(incoming_df)

    print("\nAll four Silver tables complete:")
    print(f"  ipl_catalog.silver.deliveries : {spark.table(SILVER_DELIVERIES_TABLE).count():,} rows")
    print(f"  ipl_catalog.silver.matches    : {spark.table(SILVER_MATCHES_TABLE).count():,} rows")
    print(f"  ipl_catalog.silver.venues     : {spark.table(SILVER_VENUES_TABLE).count():,} rows")
    print(f"  ipl_catalog.silver.players    : {spark.table(SILVER_PLAYERS_TABLE).count():,} rows")


# run everything
run_silver_transforms()

# Testing

In [0]:
# fix broken table issue
spark.sql("DESCRIBE TABLE ipl_catalog.silver.players").show(truncate=False)

In [0]:
spark.sql("DROP TABLE IF EXISTS ipl_catalog.silver.players")
print("Dropped broken table")

# Cell 2 — Verify gone
print(spark.catalog.tableExists("ipl_catalog.silver.players"))
# Should print False